In [7]:
import torch
import cutlass 
import cutlass.cute as cute

In [1]:
class gemm_config:
    def __init__(self,
                 ACC_PER_WARP_M,
                 ACC_PER_WARP_N,
                 WARP_K_STAGES,
                 WARPS_PER_BLOCK_M,
                 WARPS_PER_BLOCK_N,
                 BK_STAGES,
                 GROUP_M,
                 GROUP_N,
                 BK=64,
                 M=8192, N=8192, K=8192):

        # ── raw inputs ────────────────────────────────────────────
        self.acc_per_warp_m    = ACC_PER_WARP_M
        self.acc_per_warp_n    = ACC_PER_WARP_N
        self.warp_k_stages     = WARP_K_STAGES
        self.warps_per_block_m = WARPS_PER_BLOCK_M
        self.warps_per_block_n = WARPS_PER_BLOCK_N
        self.bk_stages         = BK_STAGES
        self.group_m           = GROUP_M
        self.group_n           = GROUP_N

        # ── problem size ──────────────────────────────────────────
        self.M = M
        self.N = N
        self.K = K

        # ── MMA atom (m16n8k16, bf16 in / f32 acc) ───────────────
        self.mma_m = 16
        self.mma_n = 8
        self.mma_k = 16

        # ── warp tile ─────────────────────────────────────────────
        self.WM = self.mma_m * self.acc_per_warp_m   # rows owned by one warp
        self.WN = self.mma_n * self.acc_per_warp_n   # cols owned by one warp
        self.BK = BK

        # ── CTA tile ─────────────────────────────────────────────
        self.BM = self.WM * self.warps_per_block_m
        self.BN = self.WN * self.warps_per_block_n

        # ── iteration counts ──────────────────────────────────────
        self.block_k_iters = self.K  // self.BK        # outer K loop
        self.warp_k_iters  = self.BK // self.mma_k     # inner register-pipeline loop

        assert self.warp_k_stages <= self.warp_k_iters, \
            "warp_k_stages must be <= warp_k_iters (BK/mma_k)"

        # ── thread counts ─────────────────────────────────────────
        self.num_warps  = self.warps_per_block_m * self.warps_per_block_n
        self.block_size = self.num_warps * 32           # no dedicated DMA warp

        # ── smem sizes (bf16 = 2 bytes) ───────────────────────────
        self.As_bytes      = self.BM * self.BK * 2
        self.Bs_bytes      = self.BN * self.BK * 2
        self.smem_overhead = 16 * self.bk_stages        # mbarrier storage
        self.shared_bytes  = (self.As_bytes + self.Bs_bytes) * self.bk_stages \
                             + self.smem_overhead

        # ── grid ─────────────────────────────────────────────────
        self.GM        = self.M // self.BM
        self.GN        = self.N // self.BN
        self.grid_size = self.GM * self.GN

        # ── block swizzle groups ──────────────────────────────────
        self.blocks_per_group = self.group_m * self.group_n
        self.G_outer_M        = self.GM // self.group_m
        self.G_outer_N        = self.GN // self.group_n

        # ── TMA / ldmatrix swizzle (based on BK row width in bytes) ──
        ld_bytes = self.BK * 2                          # one row of A or B tile
        if   ld_bytes == 32:
            self.swizzle_mode = "32B";  self.swizzle_num = 128
        elif ld_bytes == 64:
            self.swizzle_mode = "64B";  self.swizzle_num = 384
        elif ld_bytes == 128:
            self.swizzle_mode = "128B"; self.swizzle_num = 896
        else:
            self.swizzle_mode = "NONE"; self.swizzle_num = 0

    def __repr__(self):
        return (
            f"gemm_config(\n"
            f"  problem      : M={self.M} N={self.N} K={self.K}\n"
            f"  CTA tile     : BM={self.BM} BN={self.BN} BK={self.BK}\n"
            f"  warp tile    : WM={self.WM} WN={self.WN}  ({self.warps_per_block_m}x{self.warps_per_block_n} warps)\n"
            f"  mma atom     : m{self.mma_m}n{self.mma_n}k{self.mma_k}  acc_per_warp=({self.acc_per_warp_m},{self.acc_per_warp_n})\n"
            f"  k-pipeline   : bk_stages={self.bk_stages}  warp_k_stages={self.warp_k_stages}  warp_k_iters={self.warp_k_iters}\n"
            f"  threads/CTA  : {self.block_size}  ({self.num_warps} warps, no DMA warp)\n"
            f"  smem         : {self.shared_bytes/1024:.1f} KB  (A={self.As_bytes}B x{self.bk_stages} + B={self.Bs_bytes}B x{self.bk_stages})\n"
            f"  grid         : {self.GM}x{self.GN} = {self.grid_size} CTAs\n"
            f"  swizzle      : {self.swizzle_mode} (swizzle_num={self.swizzle_num})\n"
            f")"
        )

In [2]:
import json

with open("../theCudaBender/matmul_prefetch_V3/5090_cfg.json") as f:
    raw = json.load(f)

cfg = gemm_config(
    ACC_PER_WARP_M    = raw["ACC_PER_WARP_M"],
    ACC_PER_WARP_N    = raw["ACC_PER_WARP_N"],
    WARP_K_STAGES     = raw["WARP_K_STAGES"],
    WARPS_PER_BLOCK_M = raw["WARPS_PER_BLOCK_M"],
    WARPS_PER_BLOCK_N = raw["WARPS_PER_BLOCK_N"],
    BK_STAGES         = raw["BK_STAGES"],
    GROUP_M           = raw["GROUP_M"],
    GROUP_N           = raw["GROUP_N"],
    BK                = raw["BLOCK_K"],
)

cfg

gemm_config(
  problem      : M=8192 N=8192 K=8192
  CTA tile     : BM=128 BN=64 BK=64
  warp tile    : WM=64 WN=32  (2x2 warps)
  mma atom     : m16n8k16  acc_per_warp=(4,4)
  k-pipeline   : bk_stages=2  warp_k_stages=2  warp_k_iters=4
  threads/CTA  : 128  (4 warps, no DMA warp)
  smem         : 48.0 KB  (A=16384B x2 + B=8192B x2)
  grid         : 64x128 = 8192 CTAs
  swizzle      : 128B (swizzle_num=896)
)